# 12 - SOTA Non-Generative Ensemble

Validation-tuned weighted ensemble over available models (no retraining required):
- LayoutLMv3 family predictions
- DiT raw image predictions
- strong classical baseline predictions

All tuning is done on validation only.


## 1) Imports and Paths


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation import compute_metrics, metrics_dict_to_frame, confusion_matrix_df, plot_confusion_matrix, classification_report_df

LABELS = ['invoice', 'form', 'resume', 'email', 'budget']
INVOICE_IDX = LABELS.index('invoice')

pred_dir = PROJECT_ROOT / 'outputs' / 'predictions'
fig_dir = PROJECT_ROOT / 'reports' / 'figures'
table_dir = PROJECT_ROOT / 'reports' / 'tables'
models_dir = PROJECT_ROOT / 'models' / 'experimental'

for p in [pred_dir, fig_dir, table_dir, models_dir]:
    p.mkdir(parents=True, exist_ok=True)


## 2) Load Candidate Prediction Files


In [ ]:
candidates = {
    'layoutlmv3': ('layoutlmv3_val_predictions.csv', 'layoutlmv3_test_predictions.csv'),
    'layoutlmv3_09b': ('layoutlmv3_09b_val_predictions.csv', 'layoutlmv3_09b_test_predictions.csv'),
    'layoutlmv3_large': ('layoutlmv3_large_val_predictions.csv', 'layoutlmv3_large_test_predictions.csv'),
    'dit_raw': ('dit_raw_val_predictions.csv', 'dit_raw_test_predictions.csv'),
    'text_02b': ('02b_text_only_plus_specialist_val_predictions.csv', '02b_text_only_plus_specialist_test_predictions.csv'),
    'text_only': ('text_only_val_predictions.csv', 'text_only_test_predictions.csv'),
}

conf_cols = [f'confidence_{lab}' for lab in LABELS]
loaded = {}

for name, (vf, tf) in candidates.items():
    vpath = pred_dir / vf
    tpath = pred_dir / tf
    if not (vpath.exists() and tpath.exists()):
        continue
    vdf = pd.read_csv(vpath)
    tdf = pd.read_csv(tpath)
    req = {'doc_id', 'true_label', 'pred_label'}
    if not req.issubset(vdf.columns) or not req.issubset(tdf.columns):
        continue
    if not all(c in vdf.columns for c in conf_cols) or not all(c in tdf.columns for c in conf_cols):
        continue
    loaded[name] = {'val': vdf, 'test': tdf}

print('Loaded candidates:', list(loaded.keys()))
if len(loaded) < 2:
    raise RuntimeError('Need at least two prediction files for ensemble. Run notebooks 09/10/11 first.')


## 3) Align Validation/Test by doc_id Across Models


In [ ]:
def normalize_probs(df):
    p = df[conf_cols].to_numpy(dtype=float)
    s = p.sum(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return p / s


# choose anchor model with maximum rows
anchor_name = max(loaded.keys(), key=lambda k: len(loaded[k]['val']))
val_anchor = loaded[anchor_name]['val'][['doc_id', 'true_label']].copy()
test_anchor = loaded[anchor_name]['test'][['doc_id', 'true_label']].copy()

# inner join to common doc_id set
for name,data in loaded.items():
    val_anchor = val_anchor.merge(data['val'][['doc_id']], on='doc_id', how='inner')
    test_anchor = test_anchor.merge(data['test'][['doc_id']], on='doc_id', how='inner')

val_anchor = val_anchor.drop_duplicates('doc_id').reset_index(drop=True)
test_anchor = test_anchor.drop_duplicates('doc_id').reset_index(drop=True)

print('Common val docs:', len(val_anchor), 'Common test docs:', len(test_anchor))

# build aligned probability tensors
model_names = sorted(list(loaded.keys()))

P_val = []
P_test = []
for name in model_names:
    v = val_anchor[['doc_id']].merge(loaded[name]['val'][['doc_id'] + conf_cols], on='doc_id', how='left')
    t = test_anchor[['doc_id']].merge(loaded[name]['test'][['doc_id'] + conf_cols], on='doc_id', how='left')
    P_val.append(normalize_probs(v))
    P_test.append(normalize_probs(t))

P_val = np.stack(P_val, axis=0)   # [M, N_val, C]
P_test = np.stack(P_test, axis=0) # [M, N_test, C]

y_val = val_anchor['true_label'].values
y_test = test_anchor['true_label'].values

print('tensor shapes:', P_val.shape, P_test.shape)


## 4) Individual Model Validation Ranking


In [ ]:
rows=[]
for i,name in enumerate(model_names):
    y_pred = np.array([LABELS[j] for j in P_val[i].argmax(axis=1)])
    m = compute_metrics(y_val, y_pred, LABELS)
    rows.append({'model_name': name, 'val_accuracy': m['accuracy'], 'val_macro_f1': m['macro_f1'], 'val_invoice_recall': m['invoice_recall']})

rank_df = pd.DataFrame(rows).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
rank_df


## 5) Weight Search on Validation (Dirichlet + Baselines)

Objective:
- maximize `macro_f1 + 0.75 * invoice_recall`
- with optional macro-F1 floor relative to best single model


In [ ]:
rng = np.random.default_rng(42)

best_single_macro = float(rank_df['val_macro_f1'].max())
macro_floor = best_single_macro - 0.003


def eval_weighted(P, w, y_true):
    p = np.tensordot(w, P, axes=(0,0))  # [N,C]
    pred = np.array([LABELS[i] for i in p.argmax(axis=1)])
    m = compute_metrics(y_true, pred, LABELS)
    return p, pred, m

search_rows=[]
best=None

# one-hot baselines (single models)
for i,name in enumerate(model_names):
    w = np.zeros(len(model_names), dtype=float); w[i]=1.0
    _, _, m = eval_weighted(P_val, w, y_val)
    score = m['macro_f1'] + 0.75 * m['invoice_recall']
    row = {'type':'single', 'weights': w.tolist(), 'score': score, 'val_macro_f1': m['macro_f1'], 'val_invoice_recall': m['invoice_recall'], 'val_accuracy': m['accuracy']}
    search_rows.append(row)
    if m['macro_f1'] >= macro_floor and (best is None or score > best['score']):
        best = {'weights': w, 'metrics': m, 'score': score}

# random dirichlet mixtures
N_SAMPLES = 3000
for _ in range(N_SAMPLES):
    w = rng.dirichlet(np.ones(len(model_names)))
    _, _, m = eval_weighted(P_val, w, y_val)
    score = m['macro_f1'] + 0.75 * m['invoice_recall']
    row = {'type':'dirichlet', 'weights': w.tolist(), 'score': score, 'val_macro_f1': m['macro_f1'], 'val_invoice_recall': m['invoice_recall'], 'val_accuracy': m['accuracy']}
    search_rows.append(row)
    if m['macro_f1'] >= macro_floor and (best is None or score > best['score']):
        best = {'weights': w, 'metrics': m, 'score': score}

search_df = pd.DataFrame(search_rows).sort_values('score', ascending=False).reset_index(drop=True)
search_df.head(15)


In [ ]:
if best is None:
    # fallback: strict floor impossible, use best score overall
    top = search_df.iloc[0]
    best = {'weights': np.array(top['weights'], dtype=float), 'metrics': {'macro_f1': top['val_macro_f1'], 'invoice_recall': top['val_invoice_recall'], 'accuracy': top['val_accuracy']}, 'score': float(top['score'])}

best


## 6) Final Test Evaluation (Single Shot)


In [ ]:
p_test_ens, y_test_ens, m_test_ens = eval_weighted(P_test, best['weights'], y_test)

# best single model on test for comparison
best_single_name = rank_df.iloc[0]['model_name']
best_single_idx = model_names.index(best_single_name)
y_test_single = np.array([LABELS[i] for i in P_test[best_single_idx].argmax(axis=1)])
m_test_single = compute_metrics(y_test, y_test_single, LABELS)

result_table = pd.concat([
    metrics_dict_to_frame(m_test_single, f'{best_single_name}_single', 'test'),
    metrics_dict_to_frame(m_test_ens, 'sota_non_generative_ensemble', 'test'),
], ignore_index=True)
result_table


In [ ]:
cm = confusion_matrix_df(y_test, y_test_ens, LABELS)
plot_confusion_matrix(cm, 'SOTA Non-Generative Ensemble (Test)', save_path=fig_dir / 'sota_ensemble_confusion_matrix.png')
cm


## 7) Save Ensemble Outputs


In [ ]:
ens_pred_df = pd.DataFrame({
    'doc_id': test_anchor['doc_id'].astype(str).tolist(),
    'true_label': y_test,
    'pred_label': y_test_ens,
    'split': 'test',
    'model_name': 'sota_non_generative_ensemble',
})
for i,lab in enumerate(LABELS):
    ens_pred_df[f'confidence_{lab}'] = p_test_ens[:, i]

ens_pred_df.to_csv(pred_dir / 'sota_ensemble_test_predictions.csv', index=False)
result_table.to_csv(table_dir / 'sota_ensemble_metrics.csv', index=False)
classification_report_df(y_test, y_test_ens, LABELS).to_csv(table_dir / 'sota_ensemble_classification_report_test.csv')
search_df.to_csv(table_dir / 'sota_ensemble_weight_search_val.csv', index=False)

selected = {
    'models': model_names,
    'weights': best['weights'].tolist(),
    'objective': 'macro_f1 + 0.75 * invoice_recall',
    'macro_floor': macro_floor,
    'best_val_metrics': best['metrics'],
}
with open(models_dir / 'sota_ensemble_selected_config.json', 'w', encoding='utf-8') as f:
    json.dump(selected, f, indent=2)

print('saved outputs for notebook 12')
